# PLP Data Analytics Assignment 3

## Engagement 2: Agent Network Data Cleaning

**Project:** Duka Sales and Agent Cleanup

### Project Objective

To inspect, identify and clean data-quality problems in agent transaction records while preserving valid transactions and documenting all cleaning decisions.


In [2]:
# Load the agent data
agents = pd.read_csv("agents.csv")
txns = pd.read_csv("agent_transactions.csv")

# Check the size of each dataset
print("Agents shape:", agents.shape)
print("Transactions shape:", txns.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'agents.csv'

In [3]:
import os
print(os.listdir())

['.config', 'sample_data']


In [4]:
import os
print(os.listdir())

['.config', 'sample_data']


In [5]:
import os

print(os.listdir("/var"))

['lib', 'backups', 'tmp', 'log', 'opt', 'run', 'cache', 'spool', 'local', 'mail', 'lock', 'colab']


In [6]:
import os

print("Current folder:", os.getcwd())
print("Files in content:", os.listdir("/content"))

Current folder: /content
Files in content: ['.config', 'sample_data']


In [7]:
import os

print(os.path.exists("/agent_transactions.csv"))
print(os.path.exists("/var/colab/agent_transactions.csv"))

True
False


In [8]:
import os

print("Generator:", os.path.exists("/generate_data.py"))
print("Agents:", os.path.exists("/agents.csv"))
print("Transactions:", os.path.exists("/agent_transactions.csv"))

Generator: True
Agents: True
Transactions: True


In [9]:
# Load the agent data
agents = pd.read_csv("/agents.csv")
txns = pd.read_csv("/agent_transactions.csv")

# Check dataset sizes
print("Agents shape:", agents.shape)
print("Transactions shape:", txns.shape)

Agents shape: (20, 3)
Transactions shape: (612, 5)


In [10]:
print("AGENT DATA — FIRST 5 ROWS")
display(agents.head())

print("TRANSACTION DATA — FIRST 5 ROWS")
display(txns.head())

AGENT DATA — FIRST 5 ROWS


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


TRANSACTION DATA — FIRST 5 ROWS


,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17


In [11]:
print("TRANSACTION DATA INFORMATION")
txns.info()

print("\nMISSING VALUES")
print(txns.isnull().sum())

TRANSACTION DATA INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    object 
 1   agent_id  612 non-null    object 
 2   txn_type  612 non-null    object 
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    object 
dtypes: float64(1), object(4)
memory usage: 24.0+ KB

MISSING VALUES
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64


In [12]:
# Count duplicate transaction IDs
duplicate_count = txns["txn_id"].duplicated().sum()

print("Duplicate transaction IDs:", duplicate_count)

Duplicate transaction IDs: 12


In [13]:
print(txns["txn_type"].value_counts(dropna=False))

txn_type
deposit       142
withdrawal    131
Deposit       128
DEPOSIT       111
Withdrawal    100
Name: count, dtype: int64


In [14]:
print("AMOUNT SUMMARY")
print(txns["amount"].describe())

print("\nAMOUNTS THAT ARE NOT NUMERIC")
print(pd.to_numeric(txns["amount"], errors="coerce").isna().sum())

AMOUNT SUMMARY
count       590.000000
mean       4537.201695
std       22836.165375
min         109.000000
25%        1086.750000
50%        1779.000000
75%        3069.750000
max      250000.000000
Name: amount, dtype: float64

AMOUNTS THAT ARE NOT NUMERIC
22


In [15]:
print("Transactions with amount of KES 250,000:")
display(txns[txns["amount"] == 250000])

Transactions with amount of KES 250,000:


,txn_id,agent_id,txn_type,amount,txn_date
306,TX00307,AG012,deposit,250000.0,2026-03-02
315,TX00316,AG014,deposit,250000.0,2026-04-01
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11
594,TX00595,AG016,Deposit,250000.0,2026-03-11


In [16]:
print("TRANSACTION DATE EXAMPLES")
print(txns["txn_date"].tail(15).to_list())

TRANSACTION DATE EXAMPLES
['2026-04-03', '2026-04-18', '2026-03-25', '2026-03-01', '2026-04-03', '2026-04-08', '2026-03-14', '2026-03-22', '2026-04-28', '2026-03-19', '2026-03-03', '2026-04-07', '2026-04-18', '2026-03-11', '2026-04-07']


In [17]:
slash_dates = txns[txns["txn_date"].str.contains("/", na=False)]

print("Dates using slashes:", len(slash_dates))
display(slash_dates.head(10))

Dates using slashes: 5


,txn_id,agent_id,txn_type,amount,txn_date
87,TX00088,AG014,deposit,7072.0,2026/04/27
128,TX00129,AG010,deposit,1620.0,2026/04/28
181,TX00182,AG003,DEPOSIT,533.0,2026/03/20
437,TX00438,AG008,DEPOSIT,1518.0,2026/03/30
571,TX00572,AG011,Withdrawal,1750.0,2026/03/04


In [18]:
known_agents = set(agents["agent_id"])

unknown_agents = txns[~txns["agent_id"].isin(known_agents)]

print("Transactions with unknown agents:", len(unknown_agents))
print(unknown_agents["agent_id"].value_counts())

Transactions with unknown agents: 8
agent_id
AG099    8
Name: count, dtype: int64


In [1]:
%%writefile /content/cleaning_pipeline.py

import pandas as pd
import numpy as np


def clean_agent_data(txns, agents):
    """
    Clean agent transaction data and return a cleaned DataFrame
    together with a log of the cleaning actions.
    """

    # Keep original data unchanged
    clean = txns.copy()
    log = {}

    # Record original row count
    log["original_rows"] = len(clean)

    # 1. Remove duplicate transaction IDs
    before = len(clean)
    clean = clean.drop_duplicates(subset="txn_id", keep="first").copy()
    log["duplicate_rows_removed"] = before - len(clean)

    # 2. Standardise transaction types
    clean["txn_type"] = clean["txn_type"].astype(str).str.strip().str.lower()
    clean["txn_type"] = clean["txn_type"].map({
        "deposit": "Deposit",
        "withdrawal": "Withdrawal"
    })

    # 3. Convert amounts to numbers
    clean["amount"] = pd.to_numeric(clean["amount"], errors="coerce")

    # Mark missing amounts
    clean["amount_imputed"] = clean["amount"].isna()

    # Fill missing amounts using the median
    median_amount = clean["amount"].median()
    clean["amount"] = clean["amount"].fillna(median_amount)

    # 4. Identify outliers using the IQR method
    q1 = clean["amount"].quantile(0.25)
    q3 = clean["amount"].quantile(0.75)
    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    clean["is_outlier"] = (
        (clean["amount"] < lower_limit) |
        (clean["amount"] > upper_limit)
    )

    # 5. Standardise transaction dates
    clean["txn_date"] = pd.to_datetime(
        clean["txn_date"], errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    # 6. Merge with agent register
    clean = clean.merge(
        agents[["agent_id", "county"]],
        on="agent_id",
        how="left"
    )

    # Record unknown agents
    log["unknown_agent_transactions"] = clean["county"].isna().sum()

    # Record final row count
    log["final_rows"] = len(clean)

    return clean, log

Writing /content/cleaning_pipeline.py


In [2]:
import sys
sys.path.append("/content")

from cleaning_pipeline import clean_agent_data

clean_df, log = clean_agent_data(txns, agents)

print("Cleaning completed!")
print(log)

NameError: name 'txns' is not defined

In [3]:
import pandas as pd

agents = pd.read_csv("/agents.csv")
txns = pd.read_csv("/agent_transactions.csv")

print("Agents:", agents.shape)
print("Transactions:", txns.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/agents.csv'

In [4]:
import os

print("Files in /content:")
print(os.listdir("/content"))

Files in /content:
['.config', '__pycache__', 'cleaning_pipeline.py', 'sample_data']


In [5]:
import pandas as pd

agents = pd.read_csv("/content/agents.csv")
txns = pd.read_csv("/content/agent_transactions.csv")

print("Agents:", agents.shape)
print("Transactions:", txns.shape)

Agents: (20, 3)
Transactions: (612, 5)


In [6]:
import sys
sys.path.append("/content")

from cleaning_pipeline import clean_agent_data

clean_df, log = clean_agent_data(txns, agents)

print("Cleaning completed!")
print(log)

Cleaning completed!
{'original_rows': 612, 'duplicate_rows_removed': 12, 'unknown_agent_transactions': np.int64(7), 'final_rows': 600}


In [7]:
print("Cleaned data shape:", clean_df.shape)

display(clean_df.head())

print("\nTransaction types:")
print(clean_df["txn_type"].value_counts())

print("\nMissing values:")
print(clean_df.isna().sum())

print("\nData types:")
print(clean_df.dtypes)

Cleaned data shape: (600, 8)


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru



Transaction types:
txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64

Missing values:
txn_id            0
agent_id          0
txn_type          0
amount            0
txn_date          5
amount_imputed    0
is_outlier        0
county            7
dtype: int64

Data types:
txn_id             object
agent_id           object
txn_type           object
amount            float64
txn_date           object
amount_imputed       bool
is_outlier           bool
county             object
dtype: object


In [8]:
from pathlib import Path

file_path = Path("/content/cleaning_pipeline.py")
code = file_path.read_text()

code = code.replace(
    'clean["txn_date"], errors="coerce"',
    'clean["txn_date"], format="mixed", errors="coerce"'
)

file_path.write_text(code)

print("Date conversion updated successfully!")

Date conversion updated successfully!


In [9]:
import importlib
import cleaning_pipeline

importlib.reload(cleaning_pipeline)

clean_df, log = cleaning_pipeline.clean_agent_data(txns, agents)

print("Cleaning completed!")
print(log)

print("\nMissing values:")
print(clean_df.isna().sum())

Cleaning completed!
{'original_rows': 612, 'duplicate_rows_removed': 12, 'unknown_agent_transactions': np.int64(7), 'final_rows': 600}

Missing values:
txn_id            0
agent_id          0
txn_type          0
amount            0
txn_date          0
amount_imputed    0
is_outlier        0
county            7
dtype: int64


In [10]:
print("Duplicate transaction IDs:",
      clean_df["txn_id"].duplicated().sum())

print("\nTransaction types:")
print(clean_df["txn_type"].unique())

print("\nNumber of imputed amounts:",
      clean_df["amount_imputed"].sum())

print("\nNumber of outliers:",
      clean_df["is_outlier"].sum())

print("\nDate format examples:")
print(clean_df["txn_date"].head())

Duplicate transaction IDs: 0

Transaction types:
['Deposit' 'Withdrawal']

Number of imputed amounts: 20

Number of outliers: 53

Date format examples:
0    2026-03-03
1    2026-04-27
2    2026-04-29
3    2026-04-24
4    2026-04-17
Name: txn_date, dtype: object


In [12]:
audit = pd.DataFrame([
    {
        "issue": "Duplicate transaction IDs",
        "dimension": "Uniqueness",
        "column": "txn_id",
        "rows affected": 12
    },
    {
        "issue": "Missing or invalid amounts",
        "dimension": "Completeness",
        "column": "amount",
        "rows affected": 22
    },
    {
        "issue": "Inconsistent transaction types",
        "dimension": "Consistency",
        "column": "txn_type",
        "rows affected": 384
    },
    {
        "issue": "Dates using inconsistent separators",
        "dimension": "Consistency",
        "column": "txn_date",
        "rows affected": 5
    },
    {
        "issue": "Potential amount outliers",
        "dimension": "Accuracy",
        "column": "amount",
        "rows affected": 53
    },
    {
        "issue": "Transactions with unknown agent IDs",
        "dimension": "Referential Integrity",
        "column": "agent_id",
        "rows affected": 8
    }
])

display(audit)

,issue,dimension,column,rows affected
0,Duplicate transaction IDs,Uniqueness,txn_id,12
1,Missing or invalid amounts,Completeness,amount,22
2,Inconsistent transaction types,Consistency,txn_type,384
3,Dates using inconsistent separators,Consistency,txn_date,5
4,Potential amount outliers,Accuracy,amount,53
5,Transactions with unknown agent IDs,Referential Integrity,agent_id,8


In [13]:
county_summary = clean_df.copy()

county_summary["county"] = county_summary["county"].fillna("Unknown")

county_summary = county_summary.groupby("county").agg(
    transaction_count=("txn_id", "count"),
    total_amount=("amount", "sum")
).reset_index()

display(county_summary)

,county,transaction_count,total_amount
0,Kiambu,70,411787.5
1,Kisumu,111,512795.0
2,Mombasa,133,517578.0
3,Nairobi,180,705626.0
4,Nakuru,99,513484.5
5,Unknown,7,27156.0


In [14]:
weekly_data = clean_df.copy()

weekly_data["week"] = (
    pd.to_datetime(weekly_data["txn_date"])
    .dt.to_period("W")
    .astype(str)
)

weekly_summary = pd.pivot_table(
    weekly_data,
    index="week",
    columns="txn_type",
    values="amount",
    aggfunc="sum",
    fill_value=0
).reset_index()

display(weekly_summary)

txn_type,week,Deposit,Withdrawal
0,2026-02-23/2026-03-01,5715.5,12726.0
1,2026-03-02/2026-03-08,411481.5,305979.0
2,2026-03-09/2026-03-15,361928.0,41097.5
3,2026-03-16/2026-03-22,100216.5,59648.5
4,2026-03-23/2026-03-29,82844.5,33158.5
5,2026-03-30/2026-04-05,341215.0,70398.0
6,2026-04-06/2026-04-12,96687.0,323996.0
7,2026-04-13/2026-04-19,80303.5,70692.5
8,2026-04-20/2026-04-26,117186.0,59373.0
9,2026-04-27/2026-05-03,76325.0,37455.5


In [15]:
import os

os.makedirs("/content/outputs", exist_ok=True)

clean_df.to_csv(
    "/content/outputs/agent_transactions_clean.csv",
    index=False
)

print("Cleaned transactions exported successfully!")

print("File exists:",
      os.path.exists("/content/outputs/agent_transactions_clean.csv"))

Cleaned transactions exported successfully!
File exists: True


In [16]:
print("FINAL DATA QUALITY CHECK")

print("Original transactions:", len(txns))
print("Cleaned transactions:", len(clean_df))
print("Duplicates remaining:", clean_df["txn_id"].duplicated().sum())
print("Amounts imputed:", clean_df["amount_imputed"].sum())
print("Outliers flagged:", clean_df["is_outlier"].sum())
print("Missing dates:", clean_df["txn_date"].isna().sum())
print("Unknown counties:", clean_df["county"].isna().sum())
print("Final columns:", list(clean_df.columns))

FINAL DATA QUALITY CHECK
Original transactions: 612
Cleaned transactions: 600
Duplicates remaining: 0
Amounts imputed: 20
Outliers flagged: 53
Missing dates: 0
Unknown counties: 7
Final columns: ['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'county']


# Agent Transactions: Data Quality Report

## 1. Completeness
The original dataset contained 22 missing or invalid transaction amounts. These were converted to missing numeric values and replaced with the median amount. After removing duplicate records, 20 amounts were imputed. All transaction dates were successfully standardised, with no missing dates remaining.

## 2. Uniqueness
The original dataset contained 12 duplicate transaction IDs. Duplicate records were removed, leaving 600 unique transactions. No duplicate transaction IDs remain.

## 3. Consistency
Transaction types appeared in different formats, including `deposit`, `DEPOSIT`, and `Deposit`. A total of 384 records required standardisation. All transaction types now use only `Deposit` or `Withdrawal`. Five dates using slashes were also converted to the ISO format `YYYY-MM-DD`.

## 4. Validity
Transaction amounts were converted into numeric values. Transaction types were restricted to the two accepted categories. Transaction dates were converted into a consistent date format. These checks improved the validity of the transaction records.

## 5. Accuracy
The IQR method identified 53 potential amount outliers. These records were flagged using the `is_outlier` column but were not deleted or changed. They should be reviewed by the business to determine whether the unusual amounts represent genuine transactions or errors.

## 6. Referential Integrity
The transaction records contained 8 transactions linked to an unknown agent ID, `AG099`. After duplicate removal, 7 such transactions remained. They were retained, with their county recorded as missing, rather than inventing agent information.

## Final Outcome
The dataset was reduced from 612 to 600 transactions. The final dataset contains no duplicate transaction IDs, no missing amounts or dates, standardised transaction types, numeric amounts, outlier flags and county information. The cleaned data was exported to `outputs/agent_transactions_clean.csv`.

## Recommendations
The business should validate agent IDs before accepting transactions, enforce consistent transaction types and date formats, and investigate flagged outliers. Transaction records and agent information should be protected through appropriate access controls and secure handling.

## 7. Privacy and Data Protection

The business should protect agent and transaction information by restricting access to authorised staff, using the data only for legitimate business purposes, and handling personal information in accordance with Kenya's Data Protection Act, 2019.